# CONDOR–FlexDC Behavior Model v2.1
## Prediction, bounded multi-start optimization, and real FlexDC validation

This notebook is the inference companion to:

```text
am_unified_model_training_wandb_colab_paths_configured_objective_flexdc_behavior_v2.ipynb
```

It uses the trained behavior-v2 checkpoint and the same feature definitions used during training.

The notebook exposes the three independent command-line tools again:

1. `am_flexdc_behavior_predict_one_v2.py`
2. `am_flexdc_behavior_optimize_one_v2.py`
3. `am_flexdc_behavior_end_to_end_eval_v2.py`

The optimizer now enforces the same workload-weight bounds as the FlexDC data-extraction wizard. A candidate cannot be sent to FlexDC with an invalid weight such as `0.00008` when the legal lower bound is `0.025`.

## 0. High-level environment and logging controls

In [ ]:
from pathlib import Path
import os
import sys

RUN_ENV = "colab"  # "colab" or "local"
WORKSPACE = Path("/content/workspace") if RUN_ENV == "colab" else Path.cwd().parent

COMDER_REPO_URL = "https://github.com/NetherMoon/CONDOR-FLEXDC.git"
FLEXDC_REPO_URL = "https://github.com/amenon871/FlexDC.git"
COMDER_BRANCH = "main"
FLEXDC_BRANCH = "main"
FORCE_FRESH_CLONE = True
TORCH_CPU_THREADS = 4

# Artifact acquisition modes:
#   "existing"   -> checkpoint files already exist under ARTIFACT_DIR
#   "upload_zip" -> upload one training-artifact ZIP in Colab
#   "path_zip"   -> extract ARTIFACT_ZIP_PATH
#   "repo_zip"   -> find an artifact ZIP under am_flexdc/models
ARTIFACT_MODE = "existing"
ARTIFACT_ZIP_PATH = Path("/content/drive/MyDrive/path/behavior_v2_artifacts.zip")

USE_WANDB = True
WANDB_MODE = "online"  # "online", "offline", or "disabled"
WANDB_PROJECT = "flexdc-condor-inference-v2"
WANDB_ENTITY = "amenon06-boston-university"
WANDB_FORCE_RELOGIN = False

print("RUN_ENV:", RUN_ENV)
print("WORKSPACE:", WORKSPACE)
print("ARTIFACT_MODE:", ARTIFACT_MODE)
print("USE_WANDB:", USE_WANDB, "WANDB_MODE:", WANDB_MODE)

## 1. Install dependencies

In [ ]:
if RUN_ENV == "colab":
    %pip install -q pandas numpy scipy scikit-learn tqdm matplotlib tabulate wandb
else:
    print("SKIP: local mode uses the existing Python environment.")

## 2. Clone or update both repositories

In [ ]:
WORKSPACE.mkdir(parents=True, exist_ok=True)
COMDER_ROOT = WORKSPACE / "comder-main"
FLEXDC_ROOT = WORKSPACE / "flexdc-sim"

if RUN_ENV == "colab":
    if FORCE_FRESH_CLONE:
        !rm -rf "$COMDER_ROOT" "$FLEXDC_ROOT"
    if not COMDER_ROOT.exists():
        !git clone --branch "$COMDER_BRANCH" "$COMDER_REPO_URL" "$COMDER_ROOT"
    if not FLEXDC_ROOT.exists():
        !git clone --branch "$FLEXDC_BRANCH" "$FLEXDC_REPO_URL" "$FLEXDC_ROOT"
else:
    # Change only these two lines for a local checkout.
    COMDER_ROOT = Path(r"C:/Users/Achuthan Menon/Desktop/Research Work/comder-main")
    FLEXDC_ROOT = Path(r"C:/Users/Achuthan Menon/Desktop/Research Work/FlexDC")

print("COMDER_ROOT:", COMDER_ROOT)
print("FLEXDC_ROOT:", FLEXDC_ROOT)

## 3. Resolve paths and check all inference source files

These are the separate test/execution tools. The notebook does not replace them; it calls them and then formats their outputs.

In [ ]:
TRAIN_DIR = COMDER_ROOT / "am_flexdc" / "train"
INFERENCE_OUTPUT_ROOT = COMDER_ROOT / "am_flexdc" / "results" / "behavior_v2_inference"
ARTIFACT_DIR = COMDER_ROOT / "am_flexdc" / "models" / "behavior_v2_inference_artifacts"

INFERENCE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PY = TRAIN_DIR / "data_center_model_flexdc_behavior_v2.py"
TRAIN_UTILS_PY = TRAIN_DIR / "am_flexdc_behavior_training_utilities_v2.py"
INFERENCE_UTILS_PY = TRAIN_DIR / "am_flexdc_behavior_inference_utilities_v2.py"
PREDICT_SCRIPT = TRAIN_DIR / "am_flexdc_behavior_predict_one_v2.py"
OPTIMIZE_SCRIPT = TRAIN_DIR / "am_flexdc_behavior_optimize_one_v2.py"
E2E_SCRIPT = TRAIN_DIR / "am_flexdc_behavior_end_to_end_eval_v2.py"
TEST_SCRIPT = TRAIN_DIR / "test_flexdc_behavior_inference_v2.py"

required_code = [
    MODEL_PY,
    TRAIN_UTILS_PY,
    INFERENCE_UTILS_PY,
    PREDICT_SCRIPT,
    OPTIMIZE_SCRIPT,
    E2E_SCRIPT,
    TEST_SCRIPT,
]
missing = [path for path in required_code if not path.exists()]
if missing:
    raise FileNotFoundError("Missing inference/training files:\n" + "\n".join(str(path) for path in missing))

if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

print("All required source files found:")
for path in required_code:
    print(" -", path.name)

## 4. Acquire or use the model artifacts

The **checkpoint alone is enough for prediction and optimization**. The heldout-predictions CSV is optional and is used only for the safety-margin calibration table.

In [ ]:
import shutil
import zipfile

if ARTIFACT_MODE == "existing":
    print("Using files already present under:", ARTIFACT_DIR)
elif ARTIFACT_MODE == "upload_zip":
    if RUN_ENV != "colab":
        raise ValueError("upload_zip mode is intended for Colab")
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError(f"Upload exactly one artifact ZIP; received {zip_names}")
    source_zip = Path(zip_names[0])
    if ARTIFACT_DIR.exists():
        shutil.rmtree(ARTIFACT_DIR)
    ARTIFACT_DIR.mkdir(parents=True)
    with zipfile.ZipFile(source_zip) as archive:
        archive.extractall(ARTIFACT_DIR)
    print("Extracted:", source_zip)
elif ARTIFACT_MODE == "path_zip":
    if not ARTIFACT_ZIP_PATH.exists():
        raise FileNotFoundError(ARTIFACT_ZIP_PATH)
    if ARTIFACT_DIR.exists():
        shutil.rmtree(ARTIFACT_DIR)
    ARTIFACT_DIR.mkdir(parents=True)
    with zipfile.ZipFile(ARTIFACT_ZIP_PATH) as archive:
        archive.extractall(ARTIFACT_DIR)
    print("Extracted:", ARTIFACT_ZIP_PATH)
elif ARTIFACT_MODE == "repo_zip":
    candidates = sorted((COMDER_ROOT / "am_flexdc" / "models").glob("*behavior_v2*artifacts*.zip"))
    if not candidates:
        raise FileNotFoundError("No behavior-v2 artifact ZIP found under am_flexdc/models")
    source_zip = candidates[-1]
    if ARTIFACT_DIR.exists():
        shutil.rmtree(ARTIFACT_DIR)
    ARTIFACT_DIR.mkdir(parents=True)
    with zipfile.ZipFile(source_zip) as archive:
        archive.extractall(ARTIFACT_DIR)
    print("Extracted:", source_zip)
else:
    raise ValueError(f"Unknown ARTIFACT_MODE: {ARTIFACT_MODE}")

print("Artifact directory contents:")
for path in sorted(ARTIFACT_DIR.glob("*")):
    print(" -", path.name)

## 5. Select checkpoints and optional prediction artifacts

In [ ]:
def exactly_one(pattern, required=True):
    matches = sorted(ARTIFACT_DIR.glob(pattern))
    if len(matches) == 1:
        return matches[0]
    if not required and len(matches) == 0:
        return None
    raise FileNotFoundError(f"Expected one match for {pattern}; found {matches}")

PRIMARY_CHECKPOINT = exactly_one("*best_feasibility.pt")
BEST_LOSS_CHECKPOINT = exactly_one("*best_loss.pt", required=False)
BEST_OBJECTIVE_CHECKPOINT = exactly_one("*best_objective.pt", required=False)
HELDOUT_PREDICTIONS_CSV = exactly_one("*best_feasibility_heldout_predictions.csv", required=False)
CHECKPOINT_COMPARISON_CSV = exactly_one("*checkpoint_comparison.csv", required=False)

print("Primary checkpoint:", PRIMARY_CHECKPOINT)
print("Best-loss checkpoint:", BEST_LOSS_CHECKPOINT)
print("Best-objective checkpoint:", BEST_OBJECTIVE_CHECKPOINT)
print("Optional heldout predictions:", HELDOUT_PREDICTIONS_CSV)

## 6. Scenario preset and all adjustable optimization parameters

### Scenario inputs

- `SCENARIO` chooses a known W1/W2 configuration or `CUSTOM`.
- `START_PBAR`, `START_R`, and `START_WEIGHTS` are an anchor and the first of the multi-start candidates. They do **not** restrict the other starts.
- `SERVER_COUNT_OVERRIDE` and `UTILIZATION_OVERRIDE` override values in the experiment INI.

### Safety limits

The real FlexDC constraints remain:

```text
p90 tracking <= 0.30
every Pj <= 0.10
```

The optimizer may use stricter **selection** limits. For example:

```text
tracking limit 0.26 = true threshold 0.30 minus margin 0.04
QoS limit 0.09 = true threshold 0.10 minus margin 0.01
```

A larger positive margin is more conservative.

### Multi-start optimization controls

| Parameter | Meaning |
|---|---|
| `MULTI_STARTS` | Number of different candidate bids optimized simultaneously |
| `OPTIMIZATION_ITERATIONS` | Number of Adam updates applied to each candidate |
| `OPTIMIZATION_LR` | Initial optimizer step size for Pbar, R, and weight logits |
| `OPTIMIZATION_MIN_LR` | Final step size after cosine decay |
| `TRACKING_PENALTY` | Strength of penalty above the chosen p90 limit |
| `QOS_PENALTY` | Strength of penalty above the chosen per-job QoS limit |
| `PENALTY_RAMP_FRACTION` | Fraction of iterations used to ramp penalties from weak to full strength |
| `TOP_K` | Number of distinct predicted-feasible candidates returned |
| `CANDIDATE_DISTANCE` | Minimum normalized distance between shortlisted candidates |
| `NEAR_EQUAL_START_FRACTION` | Fraction of starts initialized near equal weights |
| `HIGH_P_LOW_R_START_FRACTION` | Fraction initialized in a generally safer high-P/low-R region |
| `RANDOM_SEED` | Makes start generation reproducible |

### Physical and weight bounds

`ENFORCE_FLEXDC_WEIGHT_BOUNDS=True` mirrors the data-extraction wizard:

```text
lower = max(0.1/J, 1/server_count)
upper = min(4/J, 1 - (J-1)*lower)
```

For J=4 and 1000 servers this becomes `[0.025, 0.925]`.

`WEIGHT_MIN` and `WEIGHT_MAX` are optional **stricter** trust-region limits. They are intersected with the automatic FlexDC bounds.

In [ ]:
SCENARIO = "W2_LU"  # W2_LU, W2_HU, W1_LU, W1_HU, CUSTOM

PRESETS = {
    "W2_LU": {
        "workload": "W2-short-qos5_4.5_4_3.5.ini",
        "experiment": "exp_traditional_iso16_servers_1000.ini",
        "utilization": 0.60,
        "pbar": 0.472128,
        "r": 0.102206,
        "weights": [0.258019617529, 0.250894690209, 0.252694830182, 0.23839086208],
    },
    "W2_HU": {
        "workload": "W2-short-qos5555.ini",
        "experiment": "exp_traditional_iso16_servers_1000.ini",
        "utilization": 0.80,
        "pbar": 0.576789,
        "r": 0.138748,
        "weights": [0.25, 0.25, 0.25, 0.25],
    },
    "W1_LU": {
        "workload": "W1-train-qos3333.ini",
        "experiment": "exp_traditional_iso16_servers_1000.ini",
        "utilization": 0.60,
        "pbar": 0.528294,
        "r": 0.234583,
        "weights": [0.374697649507, 0.204404306765, 0.203905723416, 0.216992320312],
    },
    "W1_HU": {
        "workload": "W1-train-qos4444.ini",
        "experiment": "exp_traditional_iso16_servers_1000.ini",
        "utilization": 0.80,
        "pbar": 0.578285,
        "r": 0.210078,
        "weights": [0.340987626001, 0.217909180778, 0.210953321559, 0.230149871662],
    },
}

if SCENARIO == "CUSTOM":
    WORKLOAD_CONFIG = FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5_4.5_4_3.5.ini"
    EXPERIMENT_CONFIG = FLEXDC_ROOT / "configs" / "experiment" / "new_iso" / "traditional_signal" / "generated_server_counts" / "exp_traditional_iso16_servers_1000.ini"
    UTILIZATION_OVERRIDE = 0.60
    START_PBAR = 0.472128
    START_R = 0.102206
    START_WEIGHTS = [0.25, 0.25, 0.25, 0.25]
else:
    preset = PRESETS[SCENARIO]
    WORKLOAD_CONFIG = FLEXDC_ROOT / "configs" / "workload" / preset["workload"]
    EXPERIMENT_CONFIG = FLEXDC_ROOT / "configs" / "experiment" / "new_iso" / "traditional_signal" / "generated_server_counts" / preset["experiment"]
    UTILIZATION_OVERRIDE = preset["utilization"]
    START_PBAR = preset["pbar"]
    START_R = preset["r"]
    START_WEIGHTS = preset["weights"]

GRADIENT_CONFIG = FLEXDC_ROOT / "configs" / "gradient_descent" / "gradient_descent_flexdc_paper_objective.ini"
CLUSTER_CONFIG = FLEXDC_ROOT / "configs" / "cluster" / "cluster.ini"
SERVER_COUNT_OVERRIDE = None

TRACKING_LIMIT = 0.26  # set None to use 0.30 - TRACKING_MARGIN
QOS_LIMIT = 0.09      # set None to use 0.10 - QOS_MARGIN
TRACKING_MARGIN = 0.04
QOS_MARGIN = 0.01

OPTIMIZATION_MODE = "margin_constrained"  # pure_objective, exact_constrained, margin_constrained
MULTI_STARTS = 512
OPTIMIZATION_ITERATIONS = 1500
OPTIMIZATION_LR = 0.03
OPTIMIZATION_MIN_LR = 5e-4
TRACKING_PENALTY = 2000.0
QOS_PENALTY = 2000.0
PENALTY_RAMP_FRACTION = 0.30
TOP_K = 5
CANDIDATE_DISTANCE = 0.03
RANDOM_SEED = 0
NEAR_EQUAL_START_FRACTION = 0.25
HIGH_P_LOW_R_START_FRACTION = 0.25
LOG_EVERY = 25

ENFORCE_FLEXDC_WEIGHT_BOUNDS = True
WEIGHT_MIN_FRACTION_OF_EQUAL = 0.10
WEIGHT_MAX_MULTIPLE_OF_EQUAL = 4.0
WEIGHT_MIN = None   # e.g. 0.15 for a stricter trust region
WEIGHT_MAX = None   # e.g. 0.45 for a stricter trust region
R_OVER_P_MAX = None # e.g. 0.60 only when intentionally testing R <= 0.6*Pbar
PBAR_MIN_OVERRIDE = None
PBAR_MAX_OVERRIDE = None
R_MIN_OVERRIDE = None
R_MAX_OVERRIDE = None

RUN_PREDICT_ONE = True
RUN_OPTIMIZE_ONLY = True
RUN_FLEXDC_VALIDATION = True
VALIDATE_STARTING_POINT = True
VALIDATION_TIMEOUT_SECONDS = 1800

RUN_NAME = f"{SCENARIO}_behavior_v2_1"
RUN_DIR = INFERENCE_OUTPUT_ROOT / RUN_NAME
PREDICT_DIR = RUN_DIR / "predict_one"
OPTIMIZE_DIR = RUN_DIR / "optimize_only"
VALIDATION_DIR = RUN_DIR / "flexdc_validation"
for directory in [RUN_DIR, PREDICT_DIR, OPTIMIZE_DIR, VALIDATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Scenario:", SCENARIO)
print("Run directory:", RUN_DIR)

## 7. W&B login and start the inference run

The W&B run is opened **before** prediction and optimization. Later cells add the prediction, trajectory, shortlist, and simulator-validation tables to the same run.

In [ ]:
wandb_run = None
if USE_WANDB and WANDB_MODE != "disabled":
    import getpass
    import wandb

    os.environ["WANDB_MODE"] = WANDB_MODE
    if WANDB_MODE == "online":
        key = os.environ.get("WANDB_API_KEY")
        if RUN_ENV == "colab" and not key:
            try:
                from google.colab import userdata
                key = userdata.get("WANDB_API_KEY")
            except Exception:
                key = None
        if key:
            os.environ["WANDB_API_KEY"] = key
            ok = wandb.login(key=key, relogin=WANDB_FORCE_RELOGIN, verify=True)
        else:
            try:
                ok = wandb.login(relogin=WANDB_FORCE_RELOGIN, verify=True)
            except Exception:
                key = getpass.getpass("Paste W&B API key: ").strip()
                os.environ["WANDB_API_KEY"] = key
                ok = wandb.login(key=key, relogin=True, verify=True)
        if not ok:
            raise RuntimeError("W&B login failed")

    wandb_run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=RUN_NAME,
        mode=WANDB_MODE,
        config={
            "scenario": SCENARIO,
            "checkpoint": str(PRIMARY_CHECKPOINT),
            "tracking_limit": TRACKING_LIMIT,
            "qos_limit": QOS_LIMIT,
            "tracking_margin": TRACKING_MARGIN,
            "qos_margin": QOS_MARGIN,
            "optimization_mode": OPTIMIZATION_MODE,
            "multi_starts": MULTI_STARTS,
            "iterations": OPTIMIZATION_ITERATIONS,
            "optimization_lr": OPTIMIZATION_LR,
            "optimization_min_lr": OPTIMIZATION_MIN_LR,
            "top_k": TOP_K,
            "enforce_flexdc_weight_bounds": ENFORCE_FLEXDC_WEIGHT_BOUNDS,
            "weight_min": WEIGHT_MIN,
            "weight_max": WEIGHT_MAX,
            "r_over_p_max": R_OVER_P_MAX,
        },
        tags=["CONDOR", "FlexDC", "behavior-v2", "inference", "top-k"],
    )
    print("W&B run:", wandb_run.url)
else:
    print("W&B disabled.")

## 8. Import utilities and run structural tests

In [ ]:
import json
import subprocess
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown, HTML

from am_flexdc_behavior_inference_utilities_v2 import (
    OptimizationSettings,
    calculate_pr_bounds,
    calculate_weight_bounds,
    dataframe_for_csv,
    load_behavior_model,
    margin_calibration_table,
    read_experiment_config,
    read_workload_config,
    resolve_effective_weight_bounds,
    resolve_safety_limits,
    run_flexdc_validation,
    validate_weight_bounds,
    write_json,
)

loaded = load_behavior_model(PRIMARY_CHECKPOINT, device_name="auto")
if not torch.cuda.is_available():
    torch.set_num_threads(TORCH_CPU_THREADS)

structural_command = [
    sys.executable,
    "-u",
    str(TEST_SCRIPT),
    "--checkpoint", str(PRIMARY_CHECKPOINT),
    "--workload-config", str(WORKLOAD_CONFIG),
    "--experiment-config", str(EXPERIMENT_CONFIG),
    "--flexdc-root", str(FLEXDC_ROOT),
    "--gradient-config", str(GRADIENT_CONFIG),
    "--cluster-config", str(CLUSTER_CONFIG),
    "--device", "auto",
    "--run-optimizer-smoke",
]
print("Running:", " ".join(structural_command))
completed = subprocess.run(structural_command, cwd=TRAIN_DIR, text=True, capture_output=True)
print("\nSTDOUT:\n", completed.stdout)
if completed.stderr.strip():
    print("\nSTDERR:\n", completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"Structural tests failed with return code {completed.returncode}")

## 9. Inspect the scenario, P/R region, safety limits, and legal weight region

In [ ]:
workload = read_workload_config(WORKLOAD_CONFIG)
experiment = read_experiment_config(
    EXPERIMENT_CONFIG,
    server_count_override=SERVER_COUNT_OVERRIDE,
    utilization_override=UTILIZATION_OVERRIDE,
)
bounds = calculate_pr_bounds(workload)
safety = resolve_safety_limits(
    loaded.constants,
    tracking_limit=TRACKING_LIMIT,
    qos_limit=QOS_LIMIT,
    tracking_margin=TRACKING_MARGIN,
    qos_margin=QOS_MARGIN,
)
settings = OptimizationSettings(
    starts=MULTI_STARTS,
    iterations=OPTIMIZATION_ITERATIONS,
    learning_rate=OPTIMIZATION_LR,
    minimum_learning_rate=OPTIMIZATION_MIN_LR,
    mode=OPTIMIZATION_MODE,
    tracking_penalty=TRACKING_PENALTY,
    qos_penalty=QOS_PENALTY,
    penalty_ramp_fraction=PENALTY_RAMP_FRACTION,
    top_k=TOP_K,
    candidate_distance=CANDIDATE_DISTANCE,
    random_seed=RANDOM_SEED,
    near_equal_start_fraction=NEAR_EQUAL_START_FRACTION,
    high_p_low_r_start_fraction=HIGH_P_LOW_R_START_FRACTION,
    enforce_flexdc_weight_bounds=ENFORCE_FLEXDC_WEIGHT_BOUNDS,
    weight_min_fraction_of_equal=WEIGHT_MIN_FRACTION_OF_EQUAL,
    weight_max_multiple_of_equal=WEIGHT_MAX_MULTIPLE_OF_EQUAL,
    weight_min=WEIGHT_MIN,
    weight_max=WEIGHT_MAX,
    r_over_p_max=R_OVER_P_MAX,
    pbar_min_override=PBAR_MIN_OVERRIDE,
    pbar_max_override=PBAR_MAX_OVERRIDE,
    r_min_override=R_MIN_OVERRIDE,
    r_max_override=R_MAX_OVERRIDE,
    log_every=LOG_EVERY,
)
weight_bounds = resolve_effective_weight_bounds(
    settings,
    job_count=workload.job_count,
    server_count=experiment.server_count,
)
validate_weight_bounds(START_WEIGHTS, weight_bounds)

scenario_table = pd.DataFrame([
    {"Setting": "Workload", "Value": Path(WORKLOAD_CONFIG).name},
    {"Setting": "Job types J", "Value": workload.job_count},
    {"Setting": "Server count", "Value": experiment.server_count},
    {"Setting": "Utilization", "Value": experiment.utilization},
    {"Setting": "Pbar range (kW/server)", "Value": f"{bounds.pbar_lower_kw_per_server:.6f} to {bounds.pbar_upper_kw_per_server:.6f}"},
    {"Setting": "Pbar + R upper bound", "Value": f"{bounds.pr_upper_kw_per_server:.6f}"},
    {"Setting": "R lower bound", "Value": f"{bounds.r_lower_kw_per_server:.6f}"},
    {"Setting": "Legal weight range", "Value": f"{weight_bounds.final_lower:.6f} to {weight_bounds.final_upper:.6f}"},
    {"Setting": "Exact p90 threshold", "Value": safety.exact_tracking_threshold},
    {"Setting": "Selection p90 limit", "Value": safety.selection_tracking_limit},
    {"Setting": "Exact QoS threshold", "Value": safety.exact_qos_threshold},
    {"Setting": "Selection QoS limit", "Value": safety.selection_qos_limit},
])

display(Markdown("### Scenario and constraints"))
display(scenario_table.style.hide(axis="index").set_properties(**{"text-align":"left"}))

## 10. Optional safety-margin calibration

This cell is skipped when the heldout-predictions CSV is not available. It does not affect model loading or optimization.

In [ ]:
if HELDOUT_PREDICTIONS_CSV is None:
    calibration = pd.DataFrame()
    print("SKIP: no heldout-predictions CSV. The checkpoint is still fully usable.")
else:
    calibration = margin_calibration_table(HELDOUT_PREDICTIONS_CSV)
    calibration.to_csv(RUN_DIR / "margin_calibration.csv", index=False)
    selected = calibration[
        np.isclose(calibration["Tracking_Limit"], safety.selection_tracking_limit)
        & np.isclose(calibration["QoS_Limit"], safety.selection_qos_limit)
    ]
    display(Markdown("### Selected safety limits on the heldout validation data"))
    display(selected.style.hide(axis="index").format(precision=4))
    display(Markdown("### Most conservative alternatives"))
    display(calibration.head(12).style.hide(axis="index").format(precision=4))
    if wandb_run is not None:
        import wandb
        wandb_run.log({"calibration/margin_table": wandb.Table(dataframe=calibration)})

## 11. Predict one configuration using the standalone script

In [ ]:
def append_optional(command, flag, value):
    if value is not None:
        command.extend([flag, str(value)])

PREDICT_JSON = PREDICT_DIR / "starting_prediction.json"
PREDICT_JOB_CSV = PREDICT_DIR / "starting_per_job_qos.csv"

predict_command = [
    sys.executable, "-u", str(PREDICT_SCRIPT),
    "--checkpoint", str(PRIMARY_CHECKPOINT),
    "--workload-config", str(WORKLOAD_CONFIG),
    "--experiment-config", str(EXPERIMENT_CONFIG),
    "--pbar", str(START_PBAR),
    "--r", str(START_R),
    "--weights", ",".join(str(x) for x in START_WEIGHTS),
    "--utilization", str(experiment.utilization),
    "--device", "auto",
    "--tracking-margin", str(TRACKING_MARGIN),
    "--qos-margin", str(QOS_MARGIN),
    "--out-json", str(PREDICT_JSON),
    "--out-job-csv", str(PREDICT_JOB_CSV),
]
append_optional(predict_command, "--tracking-limit", TRACKING_LIMIT)
append_optional(predict_command, "--qos-limit", QOS_LIMIT)
append_optional(predict_command, "--r-over-p-max", R_OVER_P_MAX)

if RUN_PREDICT_ONE:
    completed = subprocess.run(predict_command, cwd=TRAIN_DIR, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr.strip():
        print("STDERR:\n", completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError("Predict-one script failed")
else:
    print("Predict-one disabled. Command would be:\n", " ".join(predict_command))

## 12. Nicely formatted starting-point prediction

In [ ]:
def pass_style(value):
    if isinstance(value, (bool, np.bool_)):
        return "background-color:#064e3b;color:white;font-weight:700" if value else "background-color:#7f1d1d;color:white;font-weight:700"
    return ""

def format_weights(values, digits=4):
    return "[" + ", ".join(f"{float(v):.{digits}f}" for v in values) + "]"

if PREDICT_JSON.exists():
    payload = json.loads(PREDICT_JSON.read_text())
    start_prediction = payload["summary"]
    start_per_job = pd.DataFrame(payload["per_job"])
    start_summary = pd.DataFrame([{
        "Configuration": "Starting configuration",
        "Pbar": start_prediction["Pbar_kw_per_server"],
        "R": start_prediction["R_kw_per_server"],
        "Pbar-R": start_prediction["Pbar_minus_R"],
        "Weights": format_weights(start_prediction["weights"]),
        "Pred mean tracking": start_prediction["Predicted_Mean_Tracking"],
        "Pred p90": start_prediction["Predicted_P90_Tracking"],
        "Pred max Pj": start_prediction["Predicted_Max_Pj"],
        "Pred M_RSR": start_prediction["Predicted_M_RSR"],
        "Pred objective": start_prediction["Predicted_Full_Objective"],
        "Exact pass": start_prediction["Exact_Both_Pass"],
        "Safety pass": start_prediction["Safety_Both_Pass"],
    }])
    display(Markdown("### Starting configuration"))
    display(
        start_summary.style.hide(axis="index")
        .format(precision=6)
        .map(pass_style, subset=["Exact pass", "Safety pass"])
    )
    display(Markdown("### Per-job QoS prediction"))
    display(start_per_job.style.hide(axis="index").format(precision=6))
    if wandb_run is not None:
        import wandb
        wandb_run.log({"prediction/starting_point": wandb.Table(dataframe=start_summary)})
        wandb_run.log({"prediction/starting_per_job": wandb.Table(dataframe=start_per_job)})
else:
    print("No predict-one output yet.")

## 13. What the optimizer does, step by step

1. **Create many starts.** Generate `MULTI_STARTS` different Pbar, R, and weight candidates. One start is the supplied anchor, some start near equal weights, some start at high P/low R, and the rest are broad random starts.
2. **Map unconstrained variables into legal values.** Sigmoid mappings enforce the P/R polytope. A bounded-simplex mapping enforces positive weights, sum-to-one, and the same lower/upper bounds as FlexDC.
3. **Build the 13 per-job and 12 global features.** Use the exact training feature functions and checkpoint standardization statistics.
4. **Run the frozen model.** The network predicts log mean tracking, log p90 tracking, and one Pj per job type.
5. **Reconstruct the FlexDC objective.** Convert log tracking back to physical values and analytically compute Mpower, Mtrack, MRSR, Ctrack, CQoS, and the full objective.
6. **Add smooth constraint penalties.** If predicted p90 or any Pj exceeds the selected limit, add a squared penalty. The penalty ramps to full strength during the first part of optimization.
7. **Backpropagate into the candidate inputs.** The trained neural-network parameters remain frozen. Gradients update only Pbar logits, R logits, and weight logits.
8. **Reduce the optimization learning rate.** Adam starts at `OPTIMIZATION_LR` and follows cosine decay to `OPTIMIZATION_MIN_LR`.
9. **Repeat for all iterations.** Every start follows its own trajectory in parallel.
10. **Filter and rank.** Retain only predicted-feasible, legal candidates; remove near-duplicates; rank by predicted objective; return the best `TOP_K` distinct points.
11. **Cross-score checkpoints.** Optionally check whether the best-loss and best-objective checkpoints also consider each candidate feasible.
12. **Validate in FlexDC.** Run the starting point and top-k candidates through the real simulator and select the lowest actual objective among candidates that truly pass.

## 14. Run the standalone optimize-only script

In [ ]:
OPT_PREFIX = RUN_NAME
optimize_command = [
    sys.executable, "-u", str(OPTIMIZE_SCRIPT),
    "--checkpoint", str(PRIMARY_CHECKPOINT),
    "--workload-config", str(WORKLOAD_CONFIG),
    "--experiment-config", str(EXPERIMENT_CONFIG),
    "--utilization", str(experiment.utilization),
    "--device", "auto",
    "--initial-pbar", str(START_PBAR),
    "--initial-r", str(START_R),
    "--initial-weights", ",".join(str(x) for x in START_WEIGHTS),
    "--mode", OPTIMIZATION_MODE,
    "--tracking-margin", str(TRACKING_MARGIN),
    "--qos-margin", str(QOS_MARGIN),
    "--starts", str(MULTI_STARTS),
    "--iterations", str(OPTIMIZATION_ITERATIONS),
    "--learning-rate", str(OPTIMIZATION_LR),
    "--minimum-learning-rate", str(OPTIMIZATION_MIN_LR),
    "--tracking-penalty", str(TRACKING_PENALTY),
    "--qos-penalty", str(QOS_PENALTY),
    "--penalty-ramp-fraction", str(PENALTY_RAMP_FRACTION),
    "--top-k", str(TOP_K),
    "--candidate-distance", str(CANDIDATE_DISTANCE),
    "--random-seed", str(RANDOM_SEED),
    "--near-equal-start-fraction", str(NEAR_EQUAL_START_FRACTION),
    "--high-p-low-r-start-fraction", str(HIGH_P_LOW_R_START_FRACTION),
    "--log-every", str(LOG_EVERY),
    "--weight-min-fraction-of-equal", str(WEIGHT_MIN_FRACTION_OF_EQUAL),
    "--weight-max-multiple-of-equal", str(WEIGHT_MAX_MULTIPLE_OF_EQUAL),
    "--out-dir", str(OPTIMIZE_DIR),
    "--output-prefix", OPT_PREFIX,
]
if ENFORCE_FLEXDC_WEIGHT_BOUNDS:
    optimize_command.append("--enforce-flexdc-weight-bounds")
else:
    optimize_command.append("--no-enforce-flexdc-weight-bounds")
append_optional(optimize_command, "--tracking-limit", TRACKING_LIMIT)
append_optional(optimize_command, "--qos-limit", QOS_LIMIT)
append_optional(optimize_command, "--weight-min", WEIGHT_MIN)
append_optional(optimize_command, "--weight-max", WEIGHT_MAX)
append_optional(optimize_command, "--r-over-p-max", R_OVER_P_MAX)
append_optional(optimize_command, "--pbar-min", PBAR_MIN_OVERRIDE)
append_optional(optimize_command, "--pbar-max", PBAR_MAX_OVERRIDE)
append_optional(optimize_command, "--r-min", R_MIN_OVERRIDE)
append_optional(optimize_command, "--r-max", R_MAX_OVERRIDE)
if BEST_LOSS_CHECKPOINT is not None:
    optimize_command.extend(["--secondary-checkpoint", str(BEST_LOSS_CHECKPOINT)])
if BEST_OBJECTIVE_CHECKPOINT is not None:
    optimize_command.extend(["--secondary-checkpoint", str(BEST_OBJECTIVE_CHECKPOINT)])

if RUN_OPTIMIZE_ONLY:
    print("Running optimize-only command. This is the longest model-only cell.")
    completed = subprocess.run(optimize_command, cwd=TRAIN_DIR, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr.strip():
        print("STDERR:\n", completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError("Optimize-only script failed")
else:
    print("Optimize-only disabled. Command would be:\n", " ".join(optimize_command))

## 15. Inspect optimization trajectory and top-k candidates

In [ ]:
import matplotlib.pyplot as plt

ALL_STARTS_CSV = OPTIMIZE_DIR / f"{OPT_PREFIX}_all_starts.csv"
TOP_K_CSV = OPTIMIZE_DIR / f"{OPT_PREFIX}_top_k.csv"
TRAJECTORY_CSV = OPTIMIZE_DIR / f"{OPT_PREFIX}_trajectory.csv"
OPT_METADATA_JSON = OPTIMIZE_DIR / f"{OPT_PREFIX}_metadata.json"

all_candidates = pd.read_csv(ALL_STARTS_CSV) if ALL_STARTS_CSV.exists() else pd.DataFrame()
top_k_candidates = pd.read_csv(TOP_K_CSV) if TOP_K_CSV.exists() else pd.DataFrame()
trajectory = pd.read_csv(TRAJECTORY_CSV) if TRAJECTORY_CSV.exists() else pd.DataFrame()

if len(trajectory):
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.plot(trajectory["Iteration"], trajectory["Best_Predicted_Objective"], label="Best predicted objective")
    ax.set_xlabel("Optimization iteration")
    ax.set_ylabel("Predicted full objective")
    ax.set_title("Multi-start optimization through the frozen surrogate")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

if len(top_k_candidates):
    display_columns = [
        "Candidate_Rank", "Pbar_kw_per_server", "R_kw_per_server", "Pbar_minus_R",
        "weights", "Weight_Min", "Weight_Max", "Weight_Bounds_Pass",
        "Predicted_P90_Tracking", "Predicted_Max_Pj", "Predicted_M_RSR",
        "Predicted_Full_Objective", "Safety_Tracking_Slack", "Safety_QoS_Slack",
        "Secondary_1_Safety_Pass", "Secondary_2_Safety_Pass",
    ]
    cols = [c for c in display_columns if c in top_k_candidates.columns]
    display(Markdown("### Top-k predicted-feasible candidates"))
    display(
        top_k_candidates[cols].style.hide(axis="index")
        .format(precision=6)
        .map(pass_style, subset=[c for c in ["Weight_Bounds_Pass", "Secondary_1_Safety_Pass", "Secondary_2_Safety_Pass"] if c in cols])
    )
else:
    print("No top-k candidate met the configured selection limits.")

if wandb_run is not None:
    import wandb
    if len(trajectory):
        wandb_run.log({"optimization/trajectory": wandb.Table(dataframe=trajectory)})
    if len(top_k_candidates):
        wandb_run.log({"optimization/top_k": wandb.Table(dataframe=top_k_candidates)})
    if len(all_candidates):
        wandb_run.summary["optimization/exact_feasible_starts"] = int(all_candidates["Exact_Both_Pass"].sum())
        wandb_run.summary["optimization/safety_feasible_starts"] = int(all_candidates["Safety_Both_Pass"].sum())
        wandb_run.summary["optimization/top_k_count"] = int(len(top_k_candidates))

## 16. Validate the starting point and top-k candidates in FlexDC

This cell validates the **already optimized** shortlist. It does not rerun gradient optimization.

In [ ]:
validation_rows = []
per_job_validation_rows = []

if RUN_FLEXDC_VALIDATION:
    validation_specs = []
    if VALIDATE_STARTING_POINT and PREDICT_JSON.exists():
        validation_specs.append(("start", 0, START_PBAR, START_R, START_WEIGHTS, start_prediction))
    for _, row in top_k_candidates.iterrows():
        weights = row["weights"] if isinstance(row["weights"], list) else json.loads(row["weights"])
        validation_specs.append((
            f"rank_{int(row['Candidate_Rank'])}",
            int(row["Candidate_Rank"]),
            float(row["Pbar_kw_per_server"]),
            float(row["R_kw_per_server"]),
            weights,
            row.to_dict(),
        ))

    for label, rank, pbar, reserve, weights, predicted in validation_specs:
        # This precheck now catches illegal weights before starting FlexDC.
        validate_weight_bounds(weights, weight_bounds)
        actual, actual_jobs = run_flexdc_validation(
            python_executable=sys.executable,
            flexdc_root=FLEXDC_ROOT,
            gradient_config=GRADIENT_CONFIG,
            experiment_config=EXPERIMENT_CONFIG,
            cluster_config=CLUSTER_CONFIG,
            workload_config=WORKLOAD_CONFIG,
            output_label=f"{RUN_NAME}_{label}",
            pbar_kw_per_server=pbar,
            r_kw_per_server=reserve,
            weights=weights,
            utilization=experiment.utilization,
            constants=loaded.constants,
            timeout_seconds=VALIDATION_TIMEOUT_SECONDS,
        )
        validation_rows.append({
            "Candidate": label,
            "Rank": rank,
            "Pbar": pbar,
            "R": reserve,
            "Pbar-R": pbar - reserve,
            "Weights": format_weights(weights),
            "Weight min": min(weights),
            "Weight max": max(weights),
            "Pred p90": predicted["Predicted_P90_Tracking"],
            "Actual p90": actual["Actual_P90_Tracking"],
            "Pred max Pj": predicted["Predicted_Max_Pj"],
            "Actual max Pj": actual["Actual_Max_Pj"],
            "Pred M_RSR": predicted["Predicted_M_RSR"],
            "Actual M_RSR": actual["Actual_M_RSR"],
            "Pred objective": predicted["Predicted_Full_Objective"],
            "Actual objective": actual["Actual_Full_Objective"],
            "Pred safety pass": bool(predicted["Safety_Both_Pass"]),
            "Actual pass": bool(actual["Actual_Both_Pass"]),
            "FlexDC output": actual["FlexDC_Output_Dir"],
        })
        jobs = actual_jobs.copy()
        jobs.insert(0, "Candidate", label)
        jobs["Job_Type"] = workload.job_names
        predicted_probs = np.asarray(predicted["Predicted_QoS_Probabilities"], dtype=float)
        jobs["Predicted_Pj"] = predicted_probs
        jobs["Pj_Error"] = predicted_probs - jobs["Actual_Pj"]
        per_job_validation_rows.append(jobs)

    validation_df = pd.DataFrame(validation_rows)
    per_job_validation_df = pd.concat(per_job_validation_rows, ignore_index=True) if per_job_validation_rows else pd.DataFrame()
    dataframe_for_csv(validation_df).to_csv(VALIDATION_DIR / "predicted_vs_actual.csv", index=False)
    dataframe_for_csv(per_job_validation_df).to_csv(VALIDATION_DIR / "per_job_predicted_vs_actual.csv", index=False)
else:
    validation_df = pd.DataFrame()
    per_job_validation_df = pd.DataFrame()
    print("FlexDC validation disabled. Set RUN_FLEXDC_VALIDATION=True after reviewing the model-only shortlist.")

## 17. Nicely formatted predicted-versus-actual output

In [ ]:
if len(validation_df):
    feasible = validation_df[validation_df["Actual pass"]].copy()
    if len(feasible):
        selected_actual = feasible.sort_values("Actual objective").iloc[0]
        banner = (
            f"Selected {selected_actual['Candidate']} as the lowest-objective candidate "
            f"that passed both real FlexDC constraints."
        )
        color = "#064e3b"
    else:
        selected_actual = None
        banner = "No FlexDC-validated candidate passed both real constraints."
        color = "#7f1d1d"
    display(HTML(f"<div style='background:{color};color:white;padding:12px;border-radius:8px;font-weight:800'>{banner}</div>"))

    display(Markdown("### Predicted versus actual candidate summary"))
    bool_cols = ["Pred safety pass", "Actual pass"]
    display(
        validation_df.style.hide(axis="index")
        .format(precision=6)
        .map(pass_style, subset=bool_cols)
    )

    display(Markdown("### Per-job QoS prediction versus FlexDC"))
    display(per_job_validation_df.style.hide(axis="index").format(precision=6))

    if selected_actual is not None:
        display(Markdown("### Selected actual-feasible configuration"))
        display(pd.DataFrame([selected_actual]).style.hide(axis="index").format(precision=6))

    if wandb_run is not None:
        import wandb
        wandb_run.log({"validation/predicted_vs_actual": wandb.Table(dataframe=dataframe_for_csv(validation_df))})
        wandb_run.log({"validation/per_job": wandb.Table(dataframe=dataframe_for_csv(per_job_validation_df))})
        wandb_run.summary["validation/actual_feasible_count"] = int(validation_df["Actual pass"].sum())
        if selected_actual is not None:
            wandb_run.summary["validation/selected_candidate"] = str(selected_actual["Candidate"])
            wandb_run.summary["validation/selected_actual_objective"] = float(selected_actual["Actual objective"])
else:
    print("No FlexDC validation table available yet.")

## 18. Optional full orchestrator command

This standalone script performs optimization and FlexDC validation in one process. It is included for automated runs. The notebook normally uses the separate optimize-only and validation cells above so you can inspect the shortlist before paying for simulator runs.

In [ ]:
E2E_OUT_DIR = RUN_DIR / "automated_e2e"
e2e_command = [
    sys.executable, "-u", str(E2E_SCRIPT),
    "--checkpoint", str(PRIMARY_CHECKPOINT),
    "--workload-config", str(WORKLOAD_CONFIG),
    "--experiment-config", str(EXPERIMENT_CONFIG),
    "--utilization", str(experiment.utilization),
    "--initial-pbar", str(START_PBAR),
    "--initial-r", str(START_R),
    "--initial-weights", ",".join(str(x) for x in START_WEIGHTS),
    "--validate-start",
    "--mode", OPTIMIZATION_MODE,
    "--tracking-margin", str(TRACKING_MARGIN),
    "--qos-margin", str(QOS_MARGIN),
    "--starts", str(MULTI_STARTS),
    "--iterations", str(OPTIMIZATION_ITERATIONS),
    "--top-k", str(TOP_K),
    "--enforce-flexdc-weight-bounds",
    "--weight-min-fraction-of-equal", str(WEIGHT_MIN_FRACTION_OF_EQUAL),
    "--weight-max-multiple-of-equal", str(WEIGHT_MAX_MULTIPLE_OF_EQUAL),
    "--flexdc-root", str(FLEXDC_ROOT),
    "--gradient-config", str(GRADIENT_CONFIG),
    "--cluster-config", str(CLUSTER_CONFIG),
    "--out-dir", str(E2E_OUT_DIR),
    "--run-name", RUN_NAME,
]
append_optional(e2e_command, "--tracking-limit", TRACKING_LIMIT)
append_optional(e2e_command, "--qos-limit", QOS_LIMIT)
append_optional(e2e_command, "--weight-min", WEIGHT_MIN)
append_optional(e2e_command, "--weight-max", WEIGHT_MAX)
append_optional(e2e_command, "--r-over-p-max", R_OVER_P_MAX)
print("Automated E2E command (not run automatically):\n")
print(" ".join(f'"{x}"' if " " in x else x for x in e2e_command))

## 19. Finish W&B and package all outputs

In [ ]:
import shutil

run_summary = {
    "scenario": SCENARIO,
    "checkpoint": str(PRIMARY_CHECKPOINT),
    "checkpoint_epoch": int(loaded.checkpoint.get("epoch", -1)),
    "workload_config": str(WORKLOAD_CONFIG),
    "experiment_config": str(EXPERIMENT_CONFIG),
    "bounds": bounds.to_dict(),
    "weight_bounds": weight_bounds.to_dict(),
    "safety": safety.to_dict(),
    "settings": settings.__dict__,
    "predict_one_ran": bool(PREDICT_JSON.exists()),
    "optimize_only_ran": bool(TOP_K_CSV.exists()),
    "flexdc_validation_ran": bool(len(validation_df)),
}
write_json(RUN_DIR / "run_summary.json", run_summary)

if wandb_run is not None:
    wandb_run.summary["checkpoint_epoch"] = int(loaded.checkpoint.get("epoch", -1))
    wandb_run.summary["effective_weight_min"] = float(weight_bounds.final_lower)
    wandb_run.summary["effective_weight_max"] = float(weight_bounds.final_upper)
    wandb_run.finish()

zip_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("Packaged:", zip_path)
print("Output files:")
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(RUN_DIR))

if RUN_ENV == "colab":
    from google.colab import files
    files.download(zip_path)